In [3]:
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from torchvision import transforms

# --- CONFIG ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = (360, 640)
MODEL_PATH = r"C:\Users\patel\Downloads\Lane_detection\yolov8n.pt" # your trained model
VIDEO_PATH = "test_video.mp4"                # replace with your video path

# --- LOAD MODEL ---
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# --- TRANSFORM ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(IMG_SIZE)
])

# --- INFERENCE FUNCTION ---
def predict_lane(frame):
    orig_h, orig_w = frame.shape[:2]
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, IMG_SIZE[::-1])
    img_tensor = transform(img_resized).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(img_tensor)
        mask = torch.sigmoid(output).squeeze().cpu().numpy()
        mask = (mask > 0.5).astype(np.uint8) * 255
        mask = cv2.resize(mask, (orig_w, orig_h))

    return mask

# --- VIDEO PROCESSING ---
cap = cv2.VideoCapture(VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("lane_detected_output.mp4", fourcc, 20.0,
                      (int(cap.get(3)), int(cap.get(4))))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    mask = predict_lane(frame)

    # Overlay mask on original frame
    overlay = frame.copy()
    overlay[mask > 128] = [0, 255, 0]  # green lane color
    output_frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)

    cv2.imshow("Lane Detection", output_frame)
    out.write(output_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("✅ Video saved as: lane_detected_output.mp4")


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL ultralytics.nn.tasks.DetectionModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])` or the `torch.serialization.safe_globals([ultralytics.nn.tasks.DetectionModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to